# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [23]:
# Import required libraries
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [24]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if gemini_api_key:
    print(f"gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API key exists and begins with {openrouter_api_key[:9]}")
else:
    print("OpenRouter API Key does not exists")

gemini API Key exists and begins AIzaSyAd
OpenRouter API key exists and begins with sk-or-v1-


In [25]:
# Connect to Gemini and OpenRouter
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
openai = OpenAI()
# Configure Models to use
MODEL_GEMINI = 'gemini-2.5-flash-lite'
# MODEL_OPENROUTER = 'poolside/laguna-m.1:free'
MODEL_OPENROUTER = 'nex-agi/nex-n2-pro:free'
MODEL_NVIDIA = 'nvidia/nemotron-3-super-120b-a12b:free'
MODEL_OLLAMA = 'gemma4:e4b'

In [26]:
# System Message to feed to the AI model
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [27]:
def chat(message, history):
    # This iterates through the history list and extracts only the role and content from the h object.
    # Gradio UI automatically sends history object without needing to specify it
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [28]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [29]:
get_ticket_price("toKyo")

Tool called for city toKyo


'The price of a ticket to toKyo is $1400'

In [30]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [31]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [32]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting our AI to use our Tool

There's some fiddly stuff to allow our AI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [33]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
        return response

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, tools=tools)
    response.choices[0]
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = gemini.chat.completions.create(model=MODEL_GEMINI, messages=messages)
    
    return response.choices[0].message.content

In [14]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [34]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tool called for city Berlin


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        print(f"arguments: {arguments}")
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        print(f"price_details: {price_details}")
        responses.append({
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(f"history {history}")
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"message in chat method: {message}")
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages)
    
    return response.choices[0].message.content

In [24]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


history []
message in chat method: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_bo40zese', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', index=0)])
arguments: {'destination_city': 'London'}
Tool called for city London
price_details: The price of a ticket to London is $799
history [{'role': 'user', 'content': 'What is the price of london'}, {'role': 'assistant', 'content': 'Your ticket price to London is $799.'}]
message in chat method: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7e27uw52', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_556vqmfn', function=Funct

In [38]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [39]:
import sqlite3


In [43]:
DB = "ticket_prices.db"
ticketDB = "airline_ticket_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS ticketprices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

with sqlite3.connect(ticketDB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS airlineticketprices (id INTEGER PRIMARY KEY, city TEXT, price REAL)')
    conn.commit()

In [46]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM ticketprices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [47]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [69]:
def set_ticket_price(city, price):
    print(f"set ticket price called with city - {city} and price - {price}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO ticketprices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"New ticket price for {city} is set to {price}"

In [70]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1520, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

set ticket price called with city - london and price - 799
set ticket price called with city - paris and price - 899
set ticket price called with city - tokyo and price - 1520
set ticket price called with city - sydney and price - 2999


In [71]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1520.0'

In [66]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## Exercise

Add a tool to set the price of a ticket!

In [72]:
# There's a particular dictionary structure that's required to describe our function:

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a ticket for the desired city.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticket_price": {
                "type": "string",
                "description": "The price of the ticket the user wants to set",
            },
            "city": {
                "type": "string",
                "description": "The name of the city for which the ticket price needs to be set"
            }
        },
        "required": ["ticket_price", "city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": set_price_function}, {"type": "function", "function": price_function}]

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        print(f"arguments: {arguments}")
        price = arguments.get('ticket_price')
        if price:
            city = arguments.get('city')
            print(f"city and price {city} - {price}")
            price_details = set_ticket_price(city, price)
            print(f"price_details: {price_details}")
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        destination_city = arguments.get('destination_city')
        if destination_city:
            price_details = get_ticket_price(destination_city)
            print(f"price_details: {price_details}")
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    print(f"history {history}")
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"message in chat method: {message}")
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages)
    
    return response.choices[0].message.content

In [73]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


history []
message in chat method: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_zrs72jzr', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function', index=0)])
arguments: {'destination_city': 'Berlin'}
DATABASE TOOL CALLED: Getting price for Berlin
price_details: Ticket price to Berlin is $1300.0
history [{'role': 'user', 'content': 'What is the price of ticket to berlin'}, {'role': 'assistant', 'content': 'The current ticket price to Berlin is $1300.0.'}]
message in chat method: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_rsri15ce', function=Function(arguments='{"city":"Berlin","ticket_price":"1250"}', name='set_ticket_price'), type='function', index=0)])
arguments: {'city': 'Berlin'

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>